In [91]:
from pyspark.sql import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()

# Define the file path
file_path = "D:/Users/Wednesday/Documents/GitHub/sniffers/data/frag_raw.csv"

try:
    # Read the CSV file into a DataFrame
    frag_raw_df = spark.read.option("header", "true").csv(file_path)
    frag_raw_df.show(10, truncate=False)

    print("\n ======================================= \n Data Types:")
    print(frag_raw_df.dtypes)
finally:
    spark.stop()

+----------------------------------------+-----------------+------+------------+---------------------------------------------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------+
|name                                    |gender           |rating|rating_count|main_accords                                                                                                   |perfumers|description                                                                                                                                                                                                                                          

In [2]:
from pyspark.sql import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()

# Define the file path
file_path = "D:/Users/Wednesday/Documents/GitHub/sniffers/data/frag_raw.csv"

try:
    frag_raw_df = spark.read.csv(file_path, header=True, inferSchema=True)
    frag_raw_df.createOrReplaceTempView("frag_raw")

    cleaned_df = spark.sql("""
        SELECT
            REGEXP_EXTRACT(url, '([a-zA-Z0-9]+)\\.html$', 1) AS id,         
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', 1), 'is a', 1) AS name,
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', -1), 'is a', 1) AS brand,
            REGEXP_EXTRACT(description, 'was launched in ([0-9]{4})', 1) AS release_year,
            CASE
                WHEN name LIKE '%for women and men' THEN 'unisex'
                WHEN name LIKE '%for women' THEN 'women'
                WHEN name LIKE '%for men' THEN 'men'
                ELSE NULL
            END AS gender,
            TRY_CAST(REGEXP_REPLACE(rating_count, ',', '') AS INT) AS rating_count,
            REPLACE(REPLACE(REPLACE(main_accords, '[', ''), ']', ''),"'", "") AS main_accords,
            lower(REGEXP_REPLACE(REGEXP_EXTRACT(description, 'Top notes are (.*?);', 1), " and", ",")) AS top_notes,
            lower(REGEXP_REPLACE(REGEXP_EXTRACT(description, 'middle notes are (.*?);', 1), " and", ",")) AS mid_notes,
            lower(REPLACE(REGEXP_REPLACE(REGEXP_EXTRACT(description, 'base notes are (.*)', 1), " and", ","), ".", "")) AS base_notes,
            url
        FROM frag_raw
    """)

    cleaned_df.show(truncate=False)

    print("Schema:")
    cleaned_df.printSchema()

    print("\n ======================================= \n Data Types:")
    print(cleaned_df.dtypes)


except Exception as e:
    print(f"An error occurred: {e}")


+-----+-----------------+--------------------+------------+------+------------+-----------------------------------------------------------------------------------------+------------------------------------------------------------------------+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+-----------------------------------------------------------------------------------+
|id   |name             |brand               |release_year|gender|rating_count|main_accords                                                                             |top_notes                                                               |mid_notes                                                                                  |base_notes                                                      |url                                                                                |
+-----+-----------------+-------

In [6]:

from pyspark.sql.functions import split, trim, to_json, col, regexp_replace

def to_json_array(df, col_name, new_col_name):
    return df.withColumn(
        new_col_name,
        to_json(
            split(
                trim(regexp_replace(col(col_name), r"[\[\]']", "")),
                r",\s*"
            )
        )
    )

# Apply to all relevant columns
cleaned_df = to_json_array(cleaned_df, "main_accords", "main_accords_json")
cleaned_df = to_json_array(cleaned_df, "top_notes", "top_notes_json")
cleaned_df = to_json_array(cleaned_df, "mid_notes", "mid_notes_json")
cleaned_df = to_json_array(cleaned_df, "base_notes", "base_notes_json")

cleaned_df.show(truncate=False)

cleaned_df.toPandas().to_csv("frag_cleaned_sh.csv", index=False)

print("Schema:")
cleaned_df.printSchema()

print("\n======================================= \n\nData Types:")
print(cleaned_df.dtypes)


+-----+-----------------+--------------------+------------+------+------------+-----------------------------------------------------------------------------------------+------------------------------------------------------------------------+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+-----------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------+
|id   |name             |brand               |release_year|gender|rating_count|main_accords                                                             

In [7]:
# Get all unique accords and notes from the JSON columns
from pyspark.sql.functions import explode, from_json, array_distinct
from pyspark.sql.types import ArrayType, StringType

# Parse JSON arrays back to Spark arrays and explode
def get_unique_from_json_col(df, json_col):
    return (
        df
        .withColumn("arr", from_json(col(json_col), ArrayType(StringType())))
        .select(explode(col("arr")).alias("item"))
        .distinct()
        .select("item")
    )

# Collect unique values from each column
main_accords_unique = get_unique_from_json_col(cleaned_df, "main_accords_json")
top_notes_unique = get_unique_from_json_col(cleaned_df, "top_notes_json")
mid_notes_unique = get_unique_from_json_col(cleaned_df, "mid_notes_json")
base_notes_unique = get_unique_from_json_col(cleaned_df, "base_notes_json")

# Union all and get unique values
all_unique = (
    main_accords_unique
    .union(top_notes_unique)
    .union(mid_notes_unique)
    .union(base_notes_unique)
    .distinct()
    .orderBy("item")
)

all_unique_list = [row.item for row in all_unique.collect()]
print("\n======================================= \n \nAll Unique Accords and Notes:")
print(all_unique_list)
print(f"Number of distinct items: {len(all_unique_list)}")


 
All Unique Accords and Notes:
['', 'Champagne', 'Pear', 'absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger', 'african orange flower', 'african violet', 'agarwood', 'agarwood (oud)', 'agave', 'agave nectar', 'aglaia', 'akigalawood', 'albizia', 'alcohol', 'aldehydes', 'aldehydic', 'aldron', 'algae', 'algerian geranium', 'allspice', 'almiscarado', 'almond', 'almond blossom', 'almond cream', 'almond milk', 'almond tree', 'almond wood', 'aloe vera', 'alpinia', 'althaea', 'aluminum', 'alumroot', 'alyssum', 'amalfi lemon', 'amaranth', 'amaretto', 'amaryllis', 'amazon lily', 'ambarado', 'amber', 'amber from tunis', 'amber oil', 'amber this perfume is the winner of awardfifi award fragrance of the year men`s nouveau niche 2005', 'amber xtreme', 'ambergris', 'ambertonic', 'amberwood', 'ambrarome', 'ambreine', 'ambretone', 'ambrette', 'ambrette (musk mallow)', 'ambrette (musk mallow) t

In [9]:
import pandas as pd
import json

# Write the unique accords and notes to a JSON file
with open("all_unique_accords_and_notes.json", "w", encoding="utf-8") as f:
    json.dump(all_unique_list, f, ensure_ascii=False, indent=2)

print("Unique accords and notes written to all_unique_accords_and_notes.json")


Unique accords and notes written to all_unique_accords_and_notes.json


In [12]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.


In [102]:
import json
import chromadb
import numpy as np

# Define the path to your JSON file and the directory for the database
JSON_FILE_PATH = r"D:/Users/Wednesday/Documents/GitHub/sniffers/all_unique_accords_and_notes.json"
CHROMA_DB_PATH = "./chroma_db"

def load_data_from_json(file_path):
    """Loads and returns the JSON data from a file."""
    with open(file_path, 'r') as f:
        return json.load(f)

def create_perfume_vectors(data):
    """
    Creates one-hot encoded vectors and prepares data for ChromaDB.
    
    In a real scenario, you would have a list of all unique notes/accords
    to create the one-hot vectors. For this example, we'll assume a simplified
    process where you already have a dictionary of perfume IDs to their vectors.
    """
    documents = []
    embeddings = []
    metadatas = []
    ids = []
    
    for item in data:
        perfume_id = item["id"]
        notes_list = item["notes"]  # Assumes your JSON has a key named "notes"
        
        # This is a simplified example. You would replace this with your
        # one-hot encoding logic based on your full notes manifest.
        # For a full implementation, you would need to build a vocabulary
        # of all possible notes and then create the vector.
        # Example: [1, 0, 1, 0, ...]
        
        # Here we'll just create a dummy vector from a list of numbers
        # to demonstrate the process.
        embedding_vector = [1.0, 0.5, 0.8]  # Replace with your actual vector
        
        # Create a combined string for the "document" field for full-text search
        combined_notes = " ".join(notes_list)
        
        documents.append(combined_notes)
        embeddings.append(embedding_vector)
        metadatas.append({"id": perfume_id, "brand": item.get("brand"), "name": item.get("name")})
        ids.append(str(perfume_id))
        
    return documents, embeddings, metadatas, ids

# Load the data
data = load_data_from_json(JSON_FILE_PATH)
documents, embeddings, metadatas, ids = create_perfume_vectors(data)

### 3. Initialize ChromaDB and Add Data

Now, use the `chromadb.PersistentClient` to create your database and load the prepared data into a collection.

```python
# Initialize a persistent client
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Create a collection. `get_or_create_collection` prevents errors if the collection already exists.
collection = client.get_or_create_collection(name="perfumes")

# Add the data to the collection
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully added {len(ids)} perfumes to the ChromaDB collection.")

### 4. Search the Database (Example)

After adding the data, you can now perform similarity searches. You'll need to generate an embedding for your query (e.g., a liked perfume's vector) and then use the `query` method.

```python
# Create a dummy query vector (a vector of a liked perfume)
query_vector = [1.0, 0.5, 0.8]

# Query the collection for the 5 most similar perfumes
results = collection.query(
    query_embeddings=[query_vector],
    n_results=5
)

# Print the results
print("\n--- Top 5 Recommended Perfumes ---")
for i, result_id in enumerate(results['ids'][0]):
    metadata = results['metadatas'][0][i]
    distance = results['distances'][0][i]
    print(f"{i+1}. Perfume ID: {result_id}, Name: {metadata.get('name')}, Brand: {metadata.get('brand')}, Distance: {distance:.4f}")

SyntaxError: unmatched ')' (705039676.py, line 78)